# 2.3 Model of the Environment: Does a World Graph Help an Agent Act in ALFWorld?

*Formerly notebook 22.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): all 274 ALFWorld games × 3 conditions,
> 26,116 calls, $2.65; Batch API, agent steps from step 18 at the standard tier (2026-11-10). Task 2.3. They are
> reported in `RESULTS-REPORT.md` and the v2 paper.

**Presentation map:** Task 2.3 (model of the environment: a graph of places, objects and their relations, updated from
observations and queried for decisions), Group 2 "Memory and state".

An agent that acts in an environment must remember what it has seen: where the key was, which drawer was empty, what
it is holding. AriGraph (Anokhin et al., IJCAI 2025) showed on a handful of TextWorld games that an LLM agent which
turns every observation into triples of a **world graph** beats agents that keep the full history, a summary or a
vector memory. Their numbers are best 3 of 5 runs on one game per task, without confidence intervals. This notebook
repeats the comparison on **ALFWorld** (Shridhar et al., ICLR 2021), the text household benchmark used by ReAct and
Reflexion, on all 274 evaluation games, paired over games.

## Frozen protocol (written before the test run)

**Research question.** Does an agent that keeps a graph of its environment act better than one that keeps a text log of
its observations?

**Hypothesis.** H1: the graph agent (c) solves more games than the vector-memory agent (b), because retrieval by
similarity drops the observations that say where things are. H2 (two-sided): the graph agent (c) differs from the
full-history agent (a); with ≤ 50 short steps the whole log fits in the prompt, so we expect no large gain here.

**Data.** ALFWorld 0.4.2 text games (TextWorld engine), the standard evaluation sets: `valid_unseen` (134 games, unseen
rooms) + `valid_seen` (140 games) = **274 games**, six task types (pick & place, examine in light, clean, heat, cool,
pick two). Fixed order: all games sorted by path, shuffled once with seed 7; the test run uses the first 10.

**Conditions** (same model `gpt-6-luna`, no reasoning, temperature 0; same system prompt; same action format; same
step limit of **50 actions**; every prompt shows the goal, the initial room description, the current observation and
the list of admissible commands):
- **(a) Full history**: every (action, observation) pair of the episode so far, verbatim.
- **(b) Vector memory**: the last 3 steps verbatim + the **6** earlier steps most similar (cosine,
  `text-embedding-3-small`) to "goal + current observation", shown in time order.
- **(c) World graph (AriGraph-style)**: the last 3 steps verbatim + the current world graph. After every step one LLM
  call turns (action, observation) into triples to **add** and triples that are **no longer true** (remove) over four
  relations (`in` object→receptacle, `at` agent→place, `holds` agent→object, `is` state: open/closed/empty/clean/hot/
  cool/…); the graph is updated before the next action. Four ontology rules keep it consistent (an object is in one
  place; the agent is at one place; open/closed exclude each other; a receptacle that receives an object is no longer
  empty). An observation of "Nothing happens." carries no information and is not sent to the updater. The whole
  current graph is the query result (ALFWorld rooms are small: ≤ ~150 triples), rendered as sorted triples.

**Primary metric.** Task success (the game reports `won`) within 50 actions; paired over games.
**Secondary.** Steps to success (games solved by both conditions of a pair), tokens per episode (all LLM calls, input +
output, incl. the graph updates), share of actions that are not admissible / have no effect.

**Analysis.** Success rate with Wilson 95% intervals per condition; paired 95% bootstrap intervals (5,000 resamples,
seed 7) over games for (c) − (a) and (c) − (b) (and (b) − (a) for reference). Verdict on the primary metric only:
"better" / "worse" if the paired interval excludes 0. Also reported by split (seen / unseen) and task type.

**n.** 274 games × 3 conditions. Expected half-width of the paired interval on a success difference: ±5–8 points
(depends on discordant pairs); the test run reports the observed width.

**Threats.** One model, one prompt, temperature 0 (one run per game, no best-of-k); the admissible-command list is
given to all conditions, which makes ALFWorld easier than in the ReAct setting (no few-shot examples here); the graph
is extracted by the same LLM and can be wrong; the vector memory uses k = 6 of at most 49 earlier steps; ALFWorld's
observations are short and templated, so the full history is cheap — the setting where AriGraph's gains were largest
(long episodes, 150 steps) is not reproduced.

**Protocol changes after the test run.** None: the 10-game test run ran end to end with the protocol as written.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import glob, hashlib, json, os, random, re, subprocess, sys, time, zipfile, urllib.request
from collections import Counter, defaultdict
from typing import Literal
import numpy as np
from pydantic import BaseModel
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/alfworld"
DATA_DIR = f"{BASE}/json_2.1.1"
VENV = ".venv-alfworld"              # separate venv: ALFWorld + TextWorld (keeps the notebook kernel clean)
PY_ALF = os.path.abspath(f"{VENV}/bin/python")
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))
N_GAMES = 10 if TEST else 274        # test mode: the first 10 games of the same fixed order
MAX_STEPS, RECENT, TOP_K = 50, 3, 6  # action limit; verbatim recent steps for (b)/(c); retrieved steps for (b)
CONDS = ("full", "vector", "graph")
print("MODE =", MODE)

MODE = batch


## The environment: ALFWorld in its own virtual environment

ALFWorld (MIT licence) needs the TextWorld engine, which the notebook kernel does not have. The next cell creates
`examples/.venv-alfworld/` with `uv` (Python 3.11, `alfworld==0.4.2`, which pulls `textworld==1.7.0`), only if it is
missing, and downloads the game files of the two evaluation splits (ALFWorld release 0.2.2 `json_2.1.1_json.zip`,
72 MB, and release 0.4.0 `json_2.1.2_tw-pddl.zip`, 36 MB; only `valid_seen` and `valid_unseen` are extracted, 122 MB).
The ALFRED trajectories and ALFWorld code are MIT-licensed. No code from the downloads is executed: the zips hold
JSON and PDDL files only.

In [2]:
if not os.path.exists(PY_ALF):
    subprocess.run(["uv", "venv", "--python", "3.11", VENV], check=True)
    subprocess.run(["uv", "pip", "install", "--python", PY_ALF, "alfworld==0.4.2", "textworld==1.7.0"], check=True)

URLS = ["https://github.com/alfworld/alfworld/releases/download/0.2.2/json_2.1.1_json.zip",
        "https://github.com/alfworld/alfworld/releases/download/0.4.0/json_2.1.2_tw-pddl.zip"]
if not glob.glob(f"{DATA_DIR}/valid_unseen/*/*/game.tw-pddl"):
    os.makedirs(BASE, exist_ok=True)
    for url in URLS:
        z = f"{BASE}/{url.rsplit('/', 1)[1]}"
        if not os.path.exists(z):
            urllib.request.urlretrieve(url, z)
        with zipfile.ZipFile(z) as f:
            keep = [m for m in f.namelist() if m.startswith(("json_2.1.1/valid_seen/", "json_2.1.1/valid_unseen/"))
                    and ".." not in m and not m.startswith("/")]
            f.extractall(BASE, members=keep)
        os.remove(z)

def load_games():
    # ALFWorld's own filter (AlfredTWEnv.collect_game_files): no movable/sliced variants, only games marked solvable.
    games = []
    for split in ("valid_unseen", "valid_seen"):
        for g in sorted(glob.glob(f"{DATA_DIR}/{split}/*/*/game.tw-pddl")):
            if "movable" in g or "Sliced" in g or not json.load(open(g)).get("solvable"):
                continue
            task_dir, trial = g.split("/")[-3:-1]
            games.append({"gid": f"{split.split('_')[1]}/{task_dir}/{trial}", "path": os.path.abspath(g),
                          "split": split.split("_")[1], "type": task_dir.split("-")[0]})
    return games

ALL_GAMES = load_games()
random.Random(7).shuffle(ALL_GAMES)
GAMES = ALL_GAMES[:N_GAMES]
print(f"{len(ALL_GAMES)} games:", dict(Counter(g["split"] for g in ALL_GAMES)), "| used:", len(GAMES))
print("task types (used):", dict(Counter(g["type"] for g in GAMES)))

274 games: {'seen': 140, 'unseen': 134} | used: 274
task types (used): {'pick_two_obj_and_place': 41, 'look_at_obj_in_light': 31, 'pick_and_place_simple': 59, 'pick_heat_then_place_in_recep': 39, 'pick_clean_then_place_in_recep': 58, 'pick_cool_then_place_in_recep': 46}


### A small JSON-lines server around the games

`alfworld_env_server.py` runs inside the ALFWorld venv. It keeps one TextWorld game per episode and answers a list of
`reset` / `step` / `close` requests per line with the observations, the admissible commands and the `won` flag. The
episodes are spread over a few server processes so that one environment step for all live episodes takes seconds. The
games are deterministic, so a replay of the notebook re-drives them from the cached LLM actions and reaches the same
states.

In [3]:
SERVER = r"""
import json, sys
proto = sys.stdout; sys.stdout = sys.stderr          # keep library prints off the protocol channel
import textworld, textworld.gym
from alfworld.agents.environment.alfred_tw_env import AlfredDemangler, AlfredInfos
RI = textworld.EnvInfos(won=True, admissible_commands=True, extras=["gamefile"])
envs = {}
def pack(o, i, d):
    return {"obs": o[0], "admissible": list(i["admissible_commands"][0]), "won": bool(i["won"][0]), "done": bool(d[0])}
for line in sys.stdin:
    out = []
    for r in json.loads(line):
        try:
            if r["op"] == "reset":
                eid = textworld.gym.register_games([r["game"]], RI, batch_size=1, asynchronous=False,
                                                   max_episode_steps=100000, wrappers=[AlfredDemangler(), AlfredInfos])
                env = textworld.gym.make(eid)
                o, i = env.reset()
                envs[r["id"]] = env
                out.append(pack(o, i, [False]))
            elif r["op"] == "step":
                o, s, d, i = envs[r["id"]].step([r["action"]])
                out.append(pack(o, i, d))
            elif r["op"] == "close":
                env = envs.pop(r["id"], None)
                if env is not None:
                    env.close()
                out.append({})
        except Exception as e:
            out.append({"error": repr(e)[:500]})
    proto.write(json.dumps(out) + "\n"); proto.flush()
"""
os.makedirs(f"{BASE}/server", exist_ok=True)
SERVER_PATH = os.path.abspath(f"{BASE}/server/alfworld_env_server.py")
open(SERVER_PATH, "w").write(SERVER.lstrip())

class EnvPool:
    def __init__(self, n=6):
        env = {**os.environ, "ALFWORLD_DATA": os.path.abspath(BASE)}
        self.log = open(f"{llm.cache}/nb22_env_server.log", "w")
        self.procs = [subprocess.Popen([PY_ALF, "-I", "-u", SERVER_PATH], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                       stderr=self.log, text=True, env=env) for _ in range(n)]
    def shard(self, eid):
        return int(hashlib.md5(eid.encode()).hexdigest(), 16) % len(self.procs)
    def call(self, reqs):
        # reqs: list of {"op", "id", ...}; returns the answers in the same order.
        groups = defaultdict(list)
        for j, r in enumerate(reqs):
            groups[self.shard(r["id"])].append(j)
        for s, js in groups.items():
            self.procs[s].stdin.write(json.dumps([reqs[j] for j in js]) + "\n"); self.procs[s].stdin.flush()
        out = [None] * len(reqs)
        for s, js in groups.items():
            for j, a in zip(js, json.loads(self.procs[s].stdout.readline())):
                if "error" in a:
                    raise RuntimeError(f"environment error for {reqs[j]['id']}: {a['error']}")
                out[j] = a
        return out
    def close(self):
        for p in self.procs:
            p.stdin.close(); p.wait(timeout=60)
        self.log.close()

pool = EnvPool()
t0 = time.time()
probe = pool.call([{"op": "reset", "id": "probe", "game": GAMES[0]["path"]}])[0]
pool.call([{"op": "close", "id": "probe"}])
print(f"environment up in {time.time() - t0:.1f} s; first game:\n{probe['obs']}\n\nadmissible: {probe['admissible'][:8]} …")

environment up in 0.8 s; first game:
-= Welcome to TextWorld, ALFRED! =-

You are in the middle of a room. Looking quickly around you, you see a bathtubbasin 1, a countertop 1, a drawer 8, a drawer 7, a drawer 6, a drawer 5, a drawer 4, a drawer 3, a drawer 2, a drawer 1, a garbagecan 1, a handtowelholder 1, a sinkbasin 1, a toilet 1, a toiletpaperhanger 1, a towelholder 2, and a towelholder 1.

Your task is to: find two tissuebox and put them in toilet.

admissible: ['go to bathtubbasin 1', 'go to countertop 1', 'go to drawer 1', 'go to drawer 2', 'go to drawer 3', 'go to drawer 4', 'go to drawer 5', 'go to drawer 6'] …


## One action prompt for all three conditions

The system prompt and the layout of the user message are the same for every condition; only the memory block differs.
Every prompt holds the goal, the initial description of the room, the current observation and the admissible commands.
The model answers in JSON with a one- or two-sentence `reasoning` and one `action`.

In [4]:
class Act(BaseModel):
    reasoning: str
    action: str

ACT_SYS = ("You are an agent in a household text game (ALFWorld). You reach the goal by issuing one text command per "
           "turn. Commands: go to <receptacle>, open <receptacle>, close <receptacle>, take <object> from <receptacle>, "
           "move <object> to <receptacle>, examine <object or receptacle>, use <lamp>, heat <object> with <microwave>, "
           "cool <object> with <fridge>, clean <object> with <sinkbasin>, inventory, look. You can carry one object at "
           "a time and must go to a receptacle before you use, open or take from it. Objects may be inside closed "
           "receptacles. The game ends as soon as the goal is reached. Use your memory of the episode to avoid "
           "searching the same place twice. Reply in JSON with a short 'reasoning' (at most two sentences) and an "
           "'action' that is exactly one of the admissible commands.")

MEMORY_HEAD = {"full": "Everything that happened in this episode so far (step: > action, then observation):",
               "vector": f"Memories retrieved from earlier in this episode (the {TOP_K} past steps most similar to the "
                         "goal and the current observation, in time order):",
               "graph": "World graph built from your observations so far (subject | relation | object; current state):"}

def step_text(i, a, o):
    return f"[{i}] > {a}\n{o}"

def memory_block(ep):
    c, steps = ep["cond"], ep["steps"]
    if c == "full":
        return MEMORY_HEAD[c] + "\n" + ("\n".join(step_text(i + 1, a, o) for i, (a, o) in enumerate(steps)) or "(none yet)")
    older = len(steps) - RECENT
    if c == "vector":
        body = "\n".join(step_text(i + 1, *steps[i]) for i in sorted(ep["retrieved"])) or "(none yet)"
    else:
        body = render_graph(ep["graph"]) or "(empty)"
    recent = "\n".join(step_text(i + 1, *steps[i]) for i in range(max(older, 0), len(steps))) or "(none yet)"
    return f"{MEMORY_HEAD[c]}\n{body}\n\nThe last {RECENT} steps:\n{recent}"

def act_body(ep, t):
    user = (f"Your task is to: {ep['goal']}\n\nInitial observation: {ep['init']}\n\n{memory_block(ep)}\n\n"
            f"Current observation: {ep['obs']}\n\nAdmissible commands: {', '.join(ep['admissible'])}\n\n"
            f"Step {t} of {MAX_STEPS}. Choose the next action.")
    return LLM.body([{"role": "system", "content": ACT_SYS}, {"role": "user", "content": user}],
                    max_tokens=200, schema=Act)

## Condition (c): observation → triples → world graph (AriGraph-style)

After each step (and once for the initial room description) one LLM call reads the current graph, the action and the
observation, and returns the triples to remove and to add. The code applies the removals, then the additions, then the
four ontology rules. The graph is a set of `(subject, relation, object)` triples with lower-case names.

In [5]:
class Triple(BaseModel):
    subject: str
    relation: Literal["in", "at", "holds", "is"]
    object: str

class GraphUpdate(BaseModel):
    remove: list[Triple]
    add: list[Triple]

GRAPH_SYS = (
    "You maintain the world graph of an agent in a household text game. The graph is a set of triples "
    "(subject, relation, object) with four relations:\n"
    "- in: an object is in or on a receptacle (apple 1 | in | fridge 1); every receptacle seen in the room is in 'room' "
    "(fridge 1 | in | room)\n"
    "- at: where the agent is (agent | at | countertop 1)\n"
    "- holds: what the agent carries (agent | holds | apple 1)\n"
    "- is: a state of an object or receptacle: open, closed, empty, visited, clean, hot, cool, on, off, sliced "
    "(drawer 2 | is | empty)\n"
    "Given the current graph, the agent's last action and the observation it produced, return the triples that are no "
    "longer true (remove) and the new triples that the observation states or the action implies (add). Use names "
    "exactly as written, with their numbers. When the agent goes somewhere, replace its 'at' triple. When it takes an "
    "object, remove the object's 'in' triple and add 'agent | holds | object'. When it moves an object to a receptacle, "
    "remove the 'holds' triple and add the object's 'in' triple. When a receptacle shows its contents, add an 'in' "
    "triple for each object and mark the receptacle 'visited' (and 'empty' if it contains nothing). When an object is "
    "heated, cooled or cleaned, add that state. Add nothing the observation does not support.")

norm = lambda s: re.sub(r"\s+", " ", str(s).lower().strip().strip(".")).strip()

def render_graph(G):
    order = {"at": 0, "holds": 1, "in": 2, "is": 3}
    return "\n".join(f"{s} | {r} | {o}" for s, r, o in sorted(G, key=lambda x: (order[x[1]], x[2], x[0])))

def apply_update(G, upd):
    G = set(G)
    for t in upd["remove"]:
        G.discard((norm(t["subject"]), t["relation"], norm(t["object"])))
    for t in upd["add"]:
        s, r, o = norm(t["subject"]), t["relation"], norm(t["object"])
        if not s or not o:
            continue
        if r in ("in", "at"):                                    # an object / the agent is in one place only
            G = {x for x in G if not (x[0] == s and x[1] == r)}
        if r == "in":                                            # a receptacle that receives an object is not empty
            G.discard((o, "is", "empty"))
        if r == "is" and o in ("open", "closed"):                # open and closed exclude each other
            G.discard((s, "is", "closed" if o == "open" else "open"))
        G.add((s, r, o))
    return G

def update_body(ep):
    a, o = ep["steps"][-1] if ep["steps"] else ("(start of the episode: look around)", ep["init"])
    user = (f"Current graph:\n{render_graph(ep['graph']) or '(empty)'}\n\nLast action: {a}\nObservation: {o}")
    return LLM.body([{"role": "system", "content": GRAPH_SYS}, {"role": "user", "content": user}],
                    max_tokens=1200, schema=GraphUpdate)

## The agent loop: one `llm.run` per step over all live episodes

Each step is (1) one graph-update call for every live graph episode whose last observation is new, (2) for the vector
episodes, the embedding of the new step and of the query, then retrieval, (3) **one** `llm.run` with the next action of
every live episode of all three conditions (one Batch job per step in batch mode, as in notebooks 1.1b/1.2b), and (4) one
environment step for all of them. An episode stops when the game reports `won` or after 50 actions.

In [6]:
# Batch mode sends each step (graph updates, then actions) as its own Batch job: up to 2 x 50 sequential jobs.
# If the Batch queue makes that too slow, set STEP_TIER = "standard" (synchronous, 2x price), as notebook 1.2b did.
STEP_TIER = "standard"   # 2026-11-10: switched from "batch" at step 18 (steps 1-17 and the step-18 graph
                         # update ran via Batch). Same model and settings; tier only. Batch steps were taking hours each.
if MODE == "batch" and STEP_TIER == "standard":
    llm.mode = "test"

def split_obs(text):
    text = re.sub(r"^-= Welcome to TextWorld, ALFRED! =-\s*", "", text.strip())
    m = re.search(r"Your task is to:\s*(.+)", text)
    return text[:m.start()].strip(), (m.group(1).strip().rstrip(".") if m else "")

EPS = {}
resets = pool.call([{"op": "reset", "id": f"{c}|{g['gid']}", "game": g["path"]} for c in CONDS for g in GAMES])
for (c, g), r in zip([(c, g) for c in CONDS for g in GAMES], resets):
    init, goal = split_obs(r["obs"])
    EPS[f"{c}|{g['gid']}"] = {"cond": c, "game": g, "goal": goal, "init": init, "obs": init, "admissible": r["admissible"],
                              "steps": [], "won": False, "done": False, "graph": set(), "graph_sizes": [],
                              "mem": [], "retrieved": [], "tok_in": 0, "tok_out": 0, "calls": 0, "noop": 0,
                              "won_at": None, "unparsed": 0}

def charge(ep, r):
    ep["tok_in"] += r.get("in", 0); ep["tok_out"] += r.get("out", 0); ep["calls"] += 1

t0 = time.time()
for t in range(1, MAX_STEPS + 1):
    live = [k for k, ep in EPS.items() if not ep["done"]]
    if not live:
        break
    # (1) graph update from the last observation (skipped when it was "Nothing happens.")
    upd = {k: update_body(EPS[k]) for k in live if EPS[k]["cond"] == "graph"
           and (not EPS[k]["steps"] or EPS[k]["steps"][-1][1].strip() != "Nothing happens.")}
    if upd:
        res = llm.run(f"nb22_graph_update_s{t:02d}", upd, parse=GraphUpdate)
        for k, r in res.items():
            charge(EPS[k], r)
            if r["parsed"] is not None:
                EPS[k]["graph"] = apply_update(EPS[k]["graph"], r["parsed"])
    for k in live:
        if EPS[k]["cond"] == "graph":
            EPS[k]["graph_sizes"].append(len(EPS[k]["graph"]))
    # (2) vector memory: embed the step that just left the recent window, retrieve the top-k older steps
    vec = [k for k in live if EPS[k]["cond"] == "vector"]
    new = [(k, len(EPS[k]["steps"]) - RECENT - 1) for k in vec if len(EPS[k]["steps"]) > RECENT]
    if new:
        E = llm.embed(f"nb22_vec_mem_s{t:02d}", [step_text(i + 1, *EPS[k]["steps"][i]) for k, i in new], cap=2000)
        for (k, i), e in zip(new, E):
            EPS[k]["mem"].append((i, e))
    qk = [k for k in vec if EPS[k]["mem"]]
    if qk:
        Q = llm.embed(f"nb22_vec_query_s{t:02d}", [f"{EPS[k]['goal']}\n{EPS[k]['obs']}" for k in qk], cap=2000)
        for k, q in zip(qk, Q):
            idx = np.array([i for i, _ in EPS[k]["mem"]]); M = np.stack([e for _, e in EPS[k]["mem"]])
            EPS[k]["retrieved"] = [int(i) for i in idx[np.argsort(-(M @ q))[:TOP_K]]]
    # (3) one LLM call per live episode, all conditions in one run
    res = llm.run(f"nb22_act_s{t:02d}", {k: act_body(EPS[k], t) for k in live}, parse=Act)
    acts = {}
    for k in live:
        r = res[k]; charge(EPS[k], r)
        if r["parsed"] is None:
            EPS[k]["unparsed"] += 1
        a = norm((r["parsed"] or {}).get("action", "")) or "look"
        acts[k] = a
    # (4) one environment step for every live episode
    outs = pool.call([{"op": "step", "id": k, "action": acts[k]} for k in live])
    for k, o in zip(live, outs):
        ep = EPS[k]
        ep["noop"] += acts[k] not in ep["admissible"] or o["obs"].strip() == "Nothing happens."
        ep["steps"].append((acts[k], o["obs"].strip()))
        ep["obs"], ep["admissible"] = o["obs"].strip(), o["admissible"]
        if o["won"]:
            ep["won"], ep["won_at"] = True, t
        if o["won"] or t == MAX_STEPS:
            ep["done"] = True
    done = sum(ep["done"] for ep in EPS.values())
    if t % 5 == 0 or done == len(EPS):
        print(f"step {t:>2}: {len(live)} live -> {sum(ep['won'] for ep in EPS.values())} won, "
              f"{len(EPS) - done} still running ({time.time() - t0:.0f} s)", flush=True)
pool.call([{"op": "close", "id": k} for k in EPS])
pool.close()

step  5: 775 live -> 89 won, 733 still running (21 s)


step 10: 596 live -> 253 won, 569 still running (38 s)


step 15: 493 live -> 345 won, 477 still running (53 s)


[nb22_act_s18] 441 new requests -> test (0 cached)


[nb22_graph_update_s19] 146 new requests -> test (0 cached)


[nb22_act_s19] 429 new requests -> test (0 cached)


... (65 outputs left out)

[nb22_act_s49] 182 new requests -> test (0 cached)


[nb22_graph_update_s50] 42 new requests -> test (0 cached)


[nb22_act_s50] 179 new requests -> test (0 cached)


step 50: 179 live -> 653 won, 0 still running (1459 s)


## Results with 95% confidence intervals

Success rate per condition (Wilson interval) and paired bootstrap intervals over games. Steps to success are compared
on the games both conditions of a pair solved. Tokens are all LLM tokens of an episode (input + output; for the graph
agent including its update calls; the vector agent's embeddings are listed in the cost ledger).

In [7]:
NAMES = {"full": "(a) full history", "vector": "(b) vector memory", "graph": "(c) world graph"}
PAIRS = [("full", "graph"), ("vector", "graph"), ("full", "vector")]
ep = lambda c, g: EPS[f"{c}|{g['gid']}"]
SUMMARY = {}

def report(label, games):
    if not games:
        return
    n = len(games)
    print(f"\n{label}  (n = {n} games)")
    print(f"{'condition':<22}{'success':>9}{'95% CI':>16}{'steps|won':>11}{'tokens/ep':>11}{'no-op acts':>12}")
    for c in CONDS:
        s = [ep(c, g)["won"] for g in games]
        lo, hi = wilson(sum(s), n)
        st = [ep(c, g)["won_at"] for g in games if ep(c, g)["won"]]
        tok = np.mean([ep(c, g)["tok_in"] + ep(c, g)["tok_out"] for g in games])
        noop = sum(ep(c, g)["noop"] for g in games) / max(sum(len(ep(c, g)["steps"]) for g in games), 1)
        SUMMARY[(label, c)] = {"n": n, "sr": np.mean(s), "ci": (lo, hi), "steps": np.mean(st) if st else float("nan"),
                               "tokens": tok, "noop": noop}
        print(f"{NAMES[c]:<22}{np.mean(s):>9.1%}{f'[{lo:.0%}, {hi:.0%}]':>16}"
              f"{(np.mean(st) if st else float('nan')):>11.1f}{tok:>11,.0f}{noop:>12.1%}")
    for a, b in PAIRS:
        d = paired_boot([float(ep(a, g)["won"]) for g in games], [float(ep(b, g)["won"]) for g in games])
        both = [g for g in games if ep(a, g)["won"] and ep(b, g)["won"]]
        ds = paired_boot([ep(a, g)["won_at"] for g in both], [ep(b, g)["won_at"] for g in both]) if both else (np.nan,) * 3
        dt = paired_boot([ep(a, g)["tok_in"] + ep(a, g)["tok_out"] for g in games],
                         [ep(b, g)["tok_in"] + ep(b, g)["tok_out"] for g in games])
        disc = sum(ep(a, g)["won"] != ep(b, g)["won"] for g in games)
        SUMMARY[(label, f"{b}-{a}")] = {"d": d, "ds": ds, "dt": dt, "both": len(both), "disc": disc}
        print(f"  {b} - {a}: success {d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}] ({disc} discordant games) | "
              f"steps on {len(both)} games won by both {ds[0]:+.1f} [{ds[1]:+.1f}, {ds[2]:+.1f}] | "
              f"tokens/ep {dt[0]:+,.0f} [{dt[1]:+,.0f}, {dt[2]:+,.0f}]")

report("all games", GAMES)
for sp in ("unseen", "seen"):
    report(f"{sp} games", [g for g in GAMES if g["split"] == sp])
print("\nSuccess by task type (games: full / vector / graph):")
for ty, n in sorted(Counter(g["type"] for g in GAMES).items()):
    gs = [g for g in GAMES if g["type"] == ty]
    print(f"  {ty:<34}{n:>4}  " + " / ".join(f"{np.mean([ep(c, g)['won'] for g in gs]):.0%}" for c in CONDS))
gr = [e for e in EPS.values() if e["cond"] == "graph"]
print(f"\nGraph agent: {np.mean([len(e['graph']) for e in gr]):.0f} triples at the end of an episode on average "
      f"(max {max(len(e['graph']) for e in gr)}); unparsed action outputs: "
      + ", ".join(f"{c} {sum(e['unparsed'] for e in EPS.values() if e['cond'] == c)}" for c in CONDS))


all games  (n = 274 games)
condition               success          95% CI  steps|won  tokens/ep  no-op acts
(a) full history          88.3%      [84%, 92%]       18.7     26,866        4.4%
(b) vector memory         63.5%      [58%, 69%]       14.3     26,691        2.9%
(c) world graph           86.5%      [82%, 90%]       19.0     54,796        3.6%
  graph - full: success -1.8% [-5.5%, +1.8%] (27 discordant games) | steps on 226 games won by both +1.1 [-0.0, +2.2] | tokens/ep +27,931 [+24,470, +31,562]
  graph - vector: success +23.0% [+17.2%, +28.8%] (83 discordant games) | steps on 164 games won by both +1.0 [-0.3, +2.5] | tokens/ep +28,106 [+24,183, +32,183]
  vector - full: success -24.8% [-30.7%, -18.6%] (90 discordant games) | steps on 163 games won by both -0.0 [-1.3, +1.3] | tokens/ep -175 [-2,436, +1,983]

unseen games  (n = 134 games)
condition               success          95% CI  steps|won  tokens/ep  no-op acts
(a) full history          94.8%      [90%, 97%]       17

### One game, three agents

The first game in which the graph agent and the full-history agent differ (or the first game), with the graph agent's
world graph at the end of the episode.

In [8]:
g = next((g for g in GAMES if ep("graph", g)["won"] != ep("full", g)["won"]), GAMES[0])
print(f"Game {g['gid']}\nGoal: {ep('full', g)['goal']}\n")
for c in CONDS:
    e = ep(c, g)
    print(f"{NAMES[c]}: {'WON at step ' + str(e['won_at']) if e['won'] else 'not solved in ' + str(len(e['steps'])) + ' steps'}")
    print("   " + " · ".join(a for a, _ in e["steps"][:25]) + (" …" if len(e["steps"]) > 25 else ""))
print("\nWorld graph of the graph agent at the end:\n" + render_graph(ep("graph", g)["graph"]))

Game seen/pick_heat_then_place_in_recep-Tomato-None-Fridge-24/trial_T20190908_033721_967359
Goal: put a hot tomato in fridge

(a) full history: not solved in 50 steps
   go to fridge 1 · open fridge 1 · go to countertop 1 · go to countertop 2 · go to countertop 3 · go to cabinet 1 · open cabinet 1 · go to cabinet 2 · open cabinet 2 · go to cabinet 3 · go to cabinet 4 · open cabinet 4 · go to cabinet 5 · open cabinet 5 · go to cabinet 6 · open cabinet 6 · go to cabinet 7 · open cabinet 7 · go to cabinet 8 · open cabinet 8 · go to cabinet 9 · open cabinet 9 · go to cabinet 10 · open cabinet 10 · go to cabinet 11 …
(b) vector memory: not solved in 50 steps
   go to countertop 1 · go to countertop 2 · go to countertop 3 · go to fridge 1 · open fridge 1 · go to cabinet 1 · open cabinet 1 · go to cabinet 2 · open cabinet 2 · go to cabinet 3 · go to cabinet 4 · open cabinet 4 · go to cabinet 5 · open cabinet 5 · go to cabinet 6 · open cabinet 6 · go to cabinet 7 · open cabinet 7 · go to cabin

## What this run cost, and the forecast for the full run

The forecast scales the measured tokens per episode of each condition to 274 games at the Batch price.
`paper/v2/cost_forecast.py` assumed 30 steps × 274 games × 3 agents (32,880 calls): $7.81 at the standard tier,
≈ $3.9 through Batch.

In [9]:
STAGES = sorted(f[:-6] for f in os.listdir(llm.cache) if f.startswith("nb22_") and f.endswith(".jsonl"))
LEDGER = llm.ledger(STAGES)
agg = defaultdict(lambda: {"calls": 0, "in": 0, "out": 0, "usd": 0.0})
for s in STAGES:
    kind = "graph update" if "graph_update" in s else "actions (all conditions)"
    for f in ("calls", "in", "out", "usd"):
        agg[kind][f] += LEDGER[s][f]
agg["embeddings (vector memory)"] = {"calls": 0, "in": LEDGER["(embeddings)"]["in"], "out": 0, "usd": LEDGER["(embeddings)"]["usd"]}
print(f"{'stage group':<30}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}")
for s, a in agg.items():
    print(f"{s:<30}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.3f}")
TOTAL_USD = sum(a["usd"] for a in agg.values())
print(f"{'TOTAL':<30}{sum(a['calls'] for a in agg.values()):>8,}{sum(a['in'] for a in agg.values()):>13,}"
      f"{sum(a['out'] for a in agg.values()):>12,}{TOTAL_USD:>8.3f}   ({len(STAGES)} stages; tier: "
      f"{'standard' if TEST else ('batch, agent steps from 18 on standard' if STEP_TIER == 'standard' else 'batch')})")

per = {c: [EPS[k] for k in EPS if EPS[k]["cond"] == c] for c in CONDS}
FORECAST = {}
for c in CONDS:
    ti, to = np.mean([e["tok_in"] for e in per[c]]), np.mean([e["tok_out"] for e in per[c]])
    calls = np.mean([e["calls"] for e in per[c]])
    FORECAST[c] = (274 * calls, 274 * (ti * 0.05 + to * 0.25) / 1e6)
    print(f"{NAMES[c]:<22} per episode: {calls:5.1f} calls, {ti:8,.0f} in, {to:6,.0f} out "
          f"-> 274 games: {274 * calls:8,.0f} calls, ${FORECAST[c][1]:.2f} via Batch")
print(f"Full-run forecast: {sum(v[0] for v in FORECAST.values()):,.0f} calls, "
      f"${sum(v[1] for v in FORECAST.values()):.2f} via Batch (${2 * sum(v[1] for v in FORECAST.values()):.2f} standard)")

stage group                      calls    input tok  output tok     USD
actions (all conditions)        19,984   22,304,952     834,838   2.017
graph update                     6,132    6,013,592     535,248   0.627
embeddings (vector memory)           0      400,392           0   0.008
TOTAL                           26,116   28,718,936   1,370,086   2.652   (100 stages; tier: batch, agent steps from 18 on standard)
(a) full history       per episode:  22.4 calls,   25,937 in,    928 out -> 274 games:    6,135 calls, $0.42 via Batch
(b) vector memory      per episode:  27.3 calls,   25,551 in,  1,140 out -> 274 games:    7,492 calls, $0.43 via Batch
(c) world graph        per episode:  45.6 calls,   51,864 in,  2,932 out -> 274 games:   12,489 calls, $0.91 via Batch
Full-run forecast: 26,116 calls, $1.76 via Batch ($3.52 standard)


## Limitations

- **Admissible commands are given.** Every condition sees the list of admissible commands, as in several recent
  LLM-agent ALFWorld setups but unlike ReAct (which used two few-shot trajectories per task type). It removes parser
  failures as a confound but makes the games easier; the absolute success rates are not comparable to ReAct's 71%.
- **Short episodes.** ≤ 50 steps of short, templated observations: the full history stays at a few thousand tokens, so
  (a) is a strong baseline. AriGraph's largest gains came from longer TextWorld games (up to 150 steps, ~14k tokens of
  history); this notebook tests the ALFWorld setting, not that one.
- **Extracted graph.** The world graph is written by the same LLM from the observations; errors in it (a missed removal,
  a wrong name) are part of the method. The four ontology rules are hand-written for this environment.
- **One run per game.** Temperature 0, one prompt, no best-of-k; AriGraph reported best 3 of 5 runs.
- **Retrieval settings.** The vector memory uses k = 6 and the last 3 steps; the values were fixed before the test run
  and not tuned.

**References.** Anokhin et al., *AriGraph: Learning Knowledge Graph World Models with Episodic Memory for LLM Agents*,
IJCAI 2025 · Shridhar et al., *ALFWorld: Aligning Text and Embodied Environments for Interactive Learning*, ICLR 2021 ·
Côté et al., *TextWorld: A Learning Environment for Text-based Games*, 2018 · Yao et al., *ReAct*, ICLR 2023 · Shinn et
al., *Reflexion*, NeurIPS 2023.

In [10]:
print("=" * 120)
print(f"RESULTS SUMMARY — Notebook 2.3: model of the environment (task 2.3), {MODE} run, gpt-6-luna (no reasoning), "
      f"ALFWorld {len(GAMES)} games, <= {MAX_STEPS} actions")
print("=" * 120)
print(f"{'subset':<14}{'condition':<22}{'success':>9}{'95% CI':>14}{'steps|won':>11}{'tokens/ep':>11}")
for label in ("all games", "unseen games", "seen games"):
    for c in CONDS:
        if (label, c) not in SUMMARY:
            continue
        s = SUMMARY[(label, c)]
        ci = f"[{s['ci'][0]:.0%}, {s['ci'][1]:.0%}]"
        print(f"{label:<14}{NAMES[c]:<22}{s['sr']:>9.1%}{ci:>14}"
              f"{s['steps']:>11.1f}{s['tokens']:>11,.0f}")
verdicts = {}
print("\nPaired differences in success over all games:")
for a, b in PAIRS:
    d = SUMMARY[("all games", f"{b}-{a}")]["d"]
    v = "better" if d[1] > 0 else "worse" if d[2] < 0 else "no resolvable difference"
    verdicts[f"{b} vs {a}"] = v
    print(f"  {NAMES[b]} - {NAMES[a]}: {d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]  -> {v}  "
          f"(interval half-width {(d[2] - d[1]) / 2:.1%})")
print(f"\nCost of this run: ${TOTAL_USD:.3f}; full-run forecast ${sum(v[1] for v in FORECAST.values()):.2f} via Batch.")
print("Verdict (primary metric, task success):", "; ".join(f"{k}: {v}" for k, v in verdicts.items() if "graph" in k))
print("\nLecture message (Task 2.3): a world graph is one way to give an agent state. Whether it beats keeping the raw"
      "\nobservation log is what the verdict line says, for short household games where the log still fits the prompt.")

RESULTS SUMMARY — Notebook 2.3: model of the environment (task 2.3), batch run, gpt-6-luna (no reasoning), ALFWorld 274 games, <= 50 actions
subset        condition               success        95% CI  steps|won  tokens/ep
all games     (a) full history          88.3%    [84%, 92%]       18.7     26,866
all games     (b) vector memory         63.5%    [58%, 69%]       14.3     26,691
all games     (c) world graph           86.5%    [82%, 90%]       19.0     54,796
unseen games  (a) full history          94.8%    [90%, 97%]       17.3     20,717
unseen games  (b) vector memory         65.7%    [57%, 73%]       12.4     24,154
unseen games  (c) world graph           91.8%    [86%, 95%]       17.9     46,173
seen games    (a) full history          82.1%    [75%, 88%]       20.3     32,751
seen games    (b) vector memory         61.4%    [53%, 69%]       16.3     29,119
seen games    (c) world graph           81.4%    [74%, 87%]       20.3     63,050

Paired differences in success over all